In [16]:
!pip install -q tiktoken datasets
try:
    import triton
    print(f'Triton already available: version {triton.__version__}')
except ImportError:
    print('Triton not found, installing...')
    !pip install -q triton

Triton already available: version 3.6.0


In [17]:
import os
os.makedirs('model', exist_ok=True)
os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('results', exist_ok=True)

In [18]:
%%writefile model/__init__.py


Overwriting model/__init__.py


In [19]:
%%writefile model/attention.py
"""
model/attention.py

Phase 4: two interchangeable causal self-attention implementations with the
exact same __init__/forward signature, so model/gpt.py's Block can swap
between them via a single flag (GPTConfig.use_fused_kernel) without changing
anything else about the model.

  - CausalSelfAttention:
        The vanilla baseline. Uses PyTorch's built-in scaled_dot_product_attention,
        which already dispatches to a fused, Flash-Attention-style kernel on
        supported GPUs (including the T4). This is a genuinely strong baseline,
        not a strawman -- which is exactly what makes beating it meaningful.

  - FusedCausalSelfAttention:
        Our own hand-written fused attention kernel, implemented in Triton.
        Forward pass only: it never materializes the full (T x T) attention
        matrix in memory, using an "online softmax" the same way FlashAttention
        does. Mathematically identical output to the vanilla version.

IMPORTANT, STATED UP FRONT RATHER THAN DISCOVERED LATER:
    A raw Triton kernel is NOT automatically differentiable -- PyTorch's autograd
    has no idea how to backprop through it unless we tell it how. Writing a
    correct, efficient FlashAttention *backward* kernel is genuinely one of the
    hardest parts of the original FlashAttention paper, and is out of scope for
    this phase. Our compromise, via a torch.autograd.Function wrapper below:

        forward()  -> our fast Triton kernel        (this is what Phase 4 benchmarks)
        backward() -> recomputed via PyTorch's SDPA  (correct, but not accelerated)

    This means: the model still trains correctly end-to-end with use_fused_kernel=True,
    but the speed win we're measuring in this phase is specifically a FORWARD-PASS
    (inference/eval) win, not a training-step win. That's a completely legitimate
    and common thing to benchmark -- it's exactly the workload Phase 5 (generation)
    and Phase 6 (quantized inference) care about -- we just don't want to overstate
    it as "training is now faster too" without the backward kernel to back that up.
    Writing that backward kernel is a natural stretch goal if there's time left.
"""

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import triton
    import triton.language as tl
    TRITON_AVAILABLE = True
except ImportError:
    TRITON_AVAILABLE = False



class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_heads

        self.qkv_proj = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        self.resid_dropout = nn.Dropout(config.dropout)
        self.dropout = config.dropout

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.out_proj(y))
        return y



if TRITON_AVAILABLE:

    @triton.jit
    def _fused_attention_kernel(
        q_ptr, k_ptr, v_ptr, out_ptr,
        stride_qb, stride_qh, stride_qt, stride_qd,
        stride_kb, stride_kh, stride_kt, stride_kd,
        stride_vb, stride_vh, stride_vt, stride_vd,
        stride_ob, stride_oh, stride_ot, stride_od,
        seq_len, head_dim,
        sm_scale,
        BLOCK_M: tl.constexpr,
        BLOCK_N: tl.constexpr,
        BLOCK_D: tl.constexpr,
    ):
       
        start_m = tl.program_id(0)
        batch_head = tl.program_id(1)

        q_ptr += batch_head * stride_qh
        k_ptr += batch_head * stride_kh
        v_ptr += batch_head * stride_vh
        out_ptr += batch_head * stride_oh

        offs_m = start_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_d = tl.arange(0, BLOCK_D)
        d_mask = offs_d < head_dim  
                                     
        q_ptrs = q_ptr + offs_m[:, None] * stride_qt + offs_d[None, :] * stride_qd
        q = tl.load(q_ptrs, mask=(offs_m[:, None] < seq_len) & d_mask[None, :], other=0.0)

      
        m_i = tl.full((BLOCK_M,), value=float("-inf"), dtype=tl.float32)
        l_i = tl.zeros((BLOCK_M,), dtype=tl.float32)
        acc = tl.zeros((BLOCK_M, BLOCK_D), dtype=tl.float32)

        
        end_n = (start_m + 1) * BLOCK_M

        for start_n in range(0, end_n, BLOCK_N):
            offs_n = start_n + tl.arange(0, BLOCK_N)

            k_ptrs = k_ptr + offs_n[:, None] * stride_kt + offs_d[None, :] * stride_kd
            k = tl.load(k_ptrs, mask=(offs_n[:, None] < seq_len) & d_mask[None, :], other=0.0)

            scores = tl.dot(q, tl.trans(k)) * sm_scale

            causal_mask = offs_m[:, None] >= offs_n[None, :]
            scores = tl.where(causal_mask, scores, float("-inf"))

            m_ij = tl.max(scores, axis=1)
            m_new = tl.maximum(m_i, m_ij)

            p = tl.exp(scores - m_new[:, None])
            alpha = tl.exp(m_i - m_new)

            l_i = l_i * alpha + tl.sum(p, axis=1)
            acc = acc * alpha[:, None]

            v_ptrs = v_ptr + offs_n[:, None] * stride_vt + offs_d[None, :] * stride_vd
            v = tl.load(v_ptrs, mask=(offs_n[:, None] < seq_len) & d_mask[None, :], other=0.0)

            acc += tl.dot(p.to(v.dtype), v)
            m_i = m_new

        acc = acc / l_i[:, None]

        out_ptrs = out_ptr + offs_m[:, None] * stride_ot + offs_d[None, :] * stride_od
        tl.store(out_ptrs, acc, mask=(offs_m[:, None] < seq_len) & d_mask[None, :])


    def _triton_fused_attention_forward(q, k, v, sm_scale):
        assert q.is_contiguous() and k.is_contiguous() and v.is_contiguous(),
        B, H, T, D = q.shape
        out = torch.empty_like(q)

        BLOCK_M = 64
        BLOCK_N = 64
        BLOCK_D = triton.next_power_of_2(D)

        grid = (triton.cdiv(T, BLOCK_M), B * H)

        _fused_attention_kernel[grid](
            q, k, v, out,
            q.stride(0), q.stride(1), q.stride(2), q.stride(3),
            k.stride(0), k.stride(1), k.stride(2), k.stride(3),
            v.stride(0), v.stride(1), v.stride(2), v.stride(3),
            out.stride(0), out.stride(1), out.stride(2), out.stride(3),
            T, D, sm_scale,
            BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_D=BLOCK_D,
        )
        return out


    class _FusedAttentionFunction(torch.autograd.Function):

        @staticmethod
        def forward(ctx, q, k, v, sm_scale):
            out = _triton_fused_attention_forward(q, k, v, sm_scale)
            ctx.save_for_backward(q, k, v)
            return out

        @staticmethod
        def backward(ctx, grad_out):
            q, k, v = ctx.saved_tensors
            with torch.enable_grad():
                q_ = q.detach().requires_grad_()
                k_ = k.detach().requires_grad_()
                v_ = v.detach().requires_grad_()
                out = F.scaled_dot_product_attention(q_, k_, v_, is_causal=True)
                grad_q, grad_k, grad_v = torch.autograd.grad(out, (q_, k_, v_), grad_out)
            return grad_q, grad_k, grad_v, None


    def triton_fused_attention(q, k, v, sm_scale):
        return _FusedAttentionFunction.apply(q, k, v, sm_scale)


class FusedCausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        if not TRITON_AVAILABLE:
            raise ImportError()

        self.n_heads = config.n_heads
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_heads
        self.sm_scale = 1.0 / math.sqrt(self.head_dim)

        self.qkv_proj = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2).contiguous()
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2).contiguous()
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2).contiguous()

        y = triton_fused_attention(q, k, v, self.sm_scale)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.out_proj(y))
        return y


Overwriting model/attention.py


In [20]:
%%writefile model/gpt.py

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

from model.attention import CausalSelfAttention, FusedCausalSelfAttention


@dataclass
class GPTConfig:
    vocab_size: int = 5000       
    context_length: int = 256    
    n_layers: int = 6
    n_heads: int = 6
    n_embd: int = 384           
    dropout: float = 0.1
    bias: bool = True           
    use_fused_kernel: bool = False 
                                    

    def __post_init__(self):
        assert self.n_embd % self.n_heads == 0, "n_embd must be divisible by n_heads"




class LayerNorm(nn.Module):

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, eps=1e-5)


class MLP(nn.Module):
   

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.fc_in = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.fc_out = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.fc_in(x)
        x = self.gelu(x)
        x = self.fc_out(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        if config.use_fused_kernel:
            self.attn = FusedCausalSelfAttention(config)
        else:
            self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x



class MiniGPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.context_length, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.ln_f = LayerNorm(config.n_embd, bias=config.bias)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.token_embedding.weight = self.lm_head.weight

        self.apply(self._init_weights)
        print(f"MiniGPT initialized: {self.num_params() / 1e6:.2f}M parameters")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, idx, targets=None):
    
        B, T = idx.shape
        assert T <= self.config.context_length, (
            f"sequence length {T} exceeds context_length {self.config.context_length}"
        )

        positions = torch.arange(0, T, dtype=torch.long, device=idx.device)

        tok_emb = self.token_embedding(idx)            
        pos_emb = self.position_embedding(positions)  
        x = self.dropout(tok_emb + pos_emb)

        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1,
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.context_length \
                else idx[:, -self.config.context_length:]

            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


if __name__ == "__main__":
    config = GPTConfig(vocab_size=1000, context_length=64, n_layers=4, n_heads=4, n_embd=128)
    model = MiniGPT(config)

    dummy_input = torch.randint(0, config.vocab_size, (2, 32))    # batch=2, seq_len=32
    dummy_targets = torch.randint(0, config.vocab_size, (2, 32))

    logits, loss = model(dummy_input, dummy_targets)
    print(f"logits shape: {logits.shape}")   # expect (2, 32, 1000)
    print(f"loss: {loss.item():.4f}")

    loss.backward()
    print("Backward pass succeeded -- model is wired up correctly.")

    generated = model.generate(dummy_input[:, :5], max_new_tokens=10)
    print(f"generated shape: {generated.shape}")   # expect (2, 15)


Overwriting model/gpt.py


In [21]:
%%writefile data/prepare_data.py

import os

import numpy as np
import tiktoken
from datasets import load_dataset


TARGET_SIZE_MB = 100        
VAL_FRACTION = 0.1          
ENCODING_NAME = "gpt2"       
OUTPUT_DIR = os.path.dirname(os.path.abspath(__file__))

TARGET_SIZE_BYTES = TARGET_SIZE_MB * 1024 * 1024


def collect_text(target_bytes):
    print(f"Streaming OpenWebText until we hit ~{target_bytes / 1024 / 1024:.0f} MB of raw text...")

   
    dataset = load_dataset(
        "Skylion007/openwebtext", split="train", streaming=True, trust_remote_code=True
    )

    chunks = []
    total_bytes = 0
    n_docs = 0

    for example in dataset:
        text = example["text"]
        chunks.append(text)
        total_bytes += len(text.encode("utf-8"))
        n_docs += 1

        if n_docs % 500 == 0:
            print(f"  ...{n_docs} docs, {total_bytes / 1024 / 1024:.1f} MB so far")

        if total_bytes >= target_bytes:
            break

    full_text = "\n\n".join(chunks)
    actual_mb = len(full_text.encode("utf-8")) / 1024 / 1024
    print(f"Done collecting: {n_docs} documents, {actual_mb:.1f} MB raw text")
    return full_text


def tokenize(text):
    enc = tiktoken.get_encoding(ENCODING_NAME)
    print(f"Tokenizing with '{ENCODING_NAME}' BPE (vocab_size={enc.n_vocab})...")

  
    ids = enc.encode_ordinary(text)
    print(f"Total tokens: {len(ids):,}")
    return ids, enc.n_vocab


def save_splits(ids, val_fraction, output_dir):
    ids = np.array(ids, dtype=np.uint16)

    n_val = int(len(ids) * val_fraction)
    train_ids = ids[:-n_val]
    val_ids = ids[-n_val:]

    train_path = os.path.join(output_dir, "train.bin")
    val_path = os.path.join(output_dir, "val.bin")

    train_ids.tofile(train_path)
    val_ids.tofile(val_path)

    print(f"train.bin: {len(train_ids):,} tokens ({os.path.getsize(train_path) / 1024 / 1024:.1f} MB)")
    print(f"val.bin:   {len(val_ids):,} tokens ({os.path.getsize(val_path) / 1024 / 1024:.1f} MB)")


if __name__ == "__main__":
    text = collect_text(TARGET_SIZE_BYTES)
    ids, vocab_size = tokenize(text)
    save_splits(ids, VAL_FRACTION, OUTPUT_DIR)

    print()
    print("Done. In model/gpt.py's GPTConfig, set:")
    print(f"    vocab_size = {vocab_size}")
    print("train.py will then memory-map train.bin / val.bin directly -- no need")
    print("to re-run this script unless you want a different data slice.")


Overwriting data/prepare_data.py


In [22]:
%%writefile config.py

import torch

from model.gpt import GPTConfig


model_config = GPTConfig(
    vocab_size=50257,     
    context_length=256,
    n_layers=6,
    n_heads=6,
    n_embd=384,
    dropout=0.1,
    bias=True,
    use_fused_kernel=False,  
)

use_amp = False                
amp_dtype = "float16"           
                                 
use_grad_checkpointing = False  
grad_accumulation_steps = 1     


batch_size = 32              
learning_rate = 3e-4
max_iters = 2000            
warmup_iters = 100
lr_decay_iters = 2000
min_lr = 3e-5
weight_decay = 0.1
grad_clip = 1.0

eval_interval = 200          
eval_iters = 50              
log_interval = 20          



data_dir = "data"            
out_dir = "checkpoints"        
device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 1337

run_name = "baseline"          

Overwriting config.py


In [23]:
%%writefile train.py

import csv
import math
import os
import time
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

import config
from model.gpt import MiniGPT



def get_batch(split):
    path = os.path.join(config.data_dir, f"{split}.bin")
    data = np.memmap(path, dtype=np.uint16, mode="r")

    ctx_len = config.model_config.context_length
    ix = torch.randint(len(data) - ctx_len, (config.batch_size,))
    x = torch.stack([
        torch.from_numpy(data[i:i + ctx_len].astype(np.int64)) for i in ix
    ])
    y = torch.stack([
        torch.from_numpy(data[i + 1:i + 1 + ctx_len].astype(np.int64)) for i in ix
    ])

    x, y = x.to(config.device), y.to(config.device)
    return x, y



def get_lr(it):
    if it < config.warmup_iters:
        return config.learning_rate * it / config.warmup_iters
    if it > config.lr_decay_iters:
        return config.min_lr
    decay_ratio = (it - config.warmup_iters) / (config.lr_decay_iters - config.warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return config.min_lr + coeff * (config.learning_rate - config.min_lr)



def autocast_ctx():
    if not config.use_amp:
        return nullcontext()
    dtype = torch.bfloat16 if config.amp_dtype == "bfloat16" else torch.float16
    return torch.autocast(device_type=config.device, dtype=dtype)



def forward_with_optional_checkpointing(model, x, y):
    if not config.use_grad_checkpointing:
        return model(x, y)

    B, T = x.shape
    positions = torch.arange(0, T, dtype=torch.long, device=x.device)
    tok_emb = model.token_embedding(x)
    pos_emb = model.position_embedding(positions)
    h = model.dropout(tok_emb + pos_emb)

    for block in model.blocks:
        h = checkpoint(block, h, use_reentrant=False)

    h = model.ln_f(h)
    logits = model.lm_head(h)
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1), ignore_index=-1)
    return logits, loss



@torch.no_grad()
def estimate_loss(model):
    model.eval()
    losses = {}
    for split in ("train", "val"):
        split_losses = torch.zeros(config.eval_iters)
        for k in range(config.eval_iters):
            x, y = get_batch(split)
            with autocast_ctx():
                _, loss = model(x, y)
            split_losses[k] = loss.item()
        losses[split] = split_losses.mean().item()
    model.train()
    return losses



def log_result(results, log_path="results/logs.csv"):
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    file_exists = os.path.isfile(log_path)

    with open(log_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(results.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(results)

    print(f"Logged result to {log_path}")


def train():
    torch.manual_seed(config.seed)
    os.makedirs(config.out_dir, exist_ok=True)

    model = MiniGPT(config.model_config).to(config.device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
    )
    
    scaler = torch.cuda.amp.GradScaler(enabled=(config.use_amp and config.amp_dtype == "float16"))

    print(f"Run: {config.run_name} | device={config.device} | "
          f"amp={config.use_amp} ({config.amp_dtype if config.use_amp else 'n/a'}) | "
          f"grad_checkpointing={config.use_grad_checkpointing} | "
          f"grad_accum_steps={config.grad_accumulation_steps}")

    t0 = time.time()
    peak_mem = 0
    losses = {"train": float("nan"), "val": float("nan")}

    for it in range(config.max_iters):
        lr = get_lr(it)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        optimizer.zero_grad(set_to_none=True)

      
        
        for _ in range(config.grad_accumulation_steps):
            x, y = get_batch("train")
            with autocast_ctx():
                _, loss = forward_with_optional_checkpointing(model, x, y)
                loss = loss / config.grad_accumulation_steps
            scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        scaler.step(optimizer)
        scaler.update()

        if config.device == "cuda":
            peak_mem = max(peak_mem, torch.cuda.max_memory_allocated() / 1024 ** 2)

        if it % config.log_interval == 0:
            print(f"iter {it:5d} | loss {loss.item() * config.grad_accumulation_steps:.4f} | lr {lr:.2e}")

        if it % config.eval_interval == 0 or it == config.max_iters - 1:
            losses = estimate_loss(model)
            elapsed = time.time() - t0
            print(f"  eval @ iter {it}: train_loss={losses['train']:.4f} "
                  f"val_loss={losses['val']:.4f} elapsed={elapsed:.1f}s peak_mem={peak_mem:.0f}MB")

    ckpt_path = os.path.join(config.out_dir, f"{config.run_name}.pt")
    torch.save({"model": model.state_dict(), "config": config.model_config}, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

    total_time = time.time() - t0
    print(f"\nDone. Total time: {total_time:.1f}s | Peak GPU memory: {peak_mem:.0f}MB")

    results = {
        "run_name": config.run_name,
        "use_amp": config.use_amp,
        "amp_dtype": config.amp_dtype if config.use_amp else "n/a",
        "use_grad_checkpointing": config.use_grad_checkpointing,
        "grad_accumulation_steps": config.grad_accumulation_steps,
        "total_time_s": round(total_time, 1),
        "peak_mem_mb": round(peak_mem, 0),
        "final_train_loss": round(losses["train"], 4),
        "final_val_loss": round(losses["val"], 4),
    }
    log_result(results)
    return results


if __name__ == "__main__":
    train()


Overwriting train.py


In [24]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/mini-gpt-project'
os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)

if os.path.isfile(f'{DRIVE_DIR}/data/train.bin'):
    !cp {DRIVE_DIR}/data/train.bin {DRIVE_DIR}/data/val.bin data/
    print('Copied existing train.bin/val.bin from Drive.')
else:
    !python data/prepare_data.py
    !cp data/train.bin data/val.bin {DRIVE_DIR}/data/
    print('Generated and backed up train.bin/val.bin.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied existing train.bin/val.bin from Drive.


In [25]:
import torch
from torch.profiler import profile, ProfilerActivity

import config
from model.gpt import MiniGPT

device = config.device
print(f'Profiling on: {device}')

model = MiniGPT(config.model_config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

from train import get_batch
x, y = get_batch('train')

for _ in range(3):
    optimizer.zero_grad(set_to_none=True)
    _, loss = model(x, y)
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()

Profiling on: cuda
MiniGPT initialized: 30.04M parameters


In [26]:
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as prof:
    optimizer.zero_grad(set_to_none=True)
    _, loss = model(x, y)
    loss.backward()
    optimizer.step()
    torch.cuda.synchronize()

print(prof.key_averages().table(sort_by='cuda_time_total', row_limit=15))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         0.28%       1.345ms         0.42%       2.040ms      39.995us     324.565ms        73.23%     324.565ms       6.364ms            51  
       autograd::engine::evaluate_function: MmBackward0         0.00%      14.913us         0.03%     144.963us     144.963us       0.000us         0.00%     146.860ms     146.860ms             1  
         

In [27]:
import copy
from model.gpt import GPTConfig

torch.manual_seed(42)
small_config_vanilla = GPTConfig(vocab_size=1000, context_length=64, n_layers=2, n_heads=4,
                                   n_embd=128, dropout=0.0, use_fused_kernel=False)
model_vanilla = MiniGPT(small_config_vanilla).to(device)

torch.manual_seed(42)
small_config_fused = GPTConfig(vocab_size=1000, context_length=64, n_layers=2, n_heads=4,
                                 n_embd=128, dropout=0.0, use_fused_kernel=True)
model_fused = MiniGPT(small_config_fused).to(device)

torch.manual_seed(0)
x_test = torch.randint(0, 1000, (2, 64), device=device)
y_test = torch.randint(0, 1000, (2, 64), device=device)

logits_vanilla, loss_vanilla = model_vanilla(x_test, y_test)
logits_fused, loss_fused = model_fused(x_test, y_test)

forward_match = torch.allclose(logits_vanilla, logits_fused, atol=1e-3, rtol=1e-3)
print(f'Forward outputs match: {forward_match}')
print(f'  vanilla loss: {loss_vanilla.item():.6f}')
print(f'  fused loss:   {loss_fused.item():.6f}')

loss_vanilla.backward()
loss_fused.backward()

grad_diffs = []
for (n1, p1), (n2, p2) in zip(model_vanilla.named_parameters(), model_fused.named_parameters()):
    if p1.grad is not None and p2.grad is not None:
        grad_diffs.append(torch.allclose(p1.grad, p2.grad, atol=1e-2, rtol=1e-2))
print(f'Gradients match for {sum(grad_diffs)}/{len(grad_diffs)} parameter tensors')

MiniGPT initialized: 0.53M parameters
MiniGPT initialized: 0.53M parameters
Forward outputs match: True
  vanilla loss: 6.917414
  fused loss:   6.917414
Gradients match for 28/28 parameter tensors


In [28]:
import time

def benchmark_forward(model, x, n_warmup=5, n_iters=20):
    model.eval()
    with torch.no_grad():
        for _ in range(n_warmup):
            model(x)
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

        start.record()
        for _ in range(n_iters):
            model(x)
        end.record()
        torch.cuda.synchronize()

        avg_ms = start.elapsed_time(end) / n_iters
        peak_mem_mb = torch.cuda.max_memory_allocated() / 1024 / 1024
    model.train()
    return avg_ms, peak_mem_mb

In [29]:
import pandas as pd

seq_lengths = [64, 128, 256]
batch_size_bench = 16
bench_rows = []

for seq_len in seq_lengths:
    cfg_kwargs = dict(vocab_size=50257, context_length=seq_len, n_layers=6, n_heads=6,
                       n_embd=384, dropout=0.0)

    torch.manual_seed(0)
    m_vanilla = MiniGPT(GPTConfig(**cfg_kwargs, use_fused_kernel=False)).to(device)
    torch.manual_seed(0)
    m_fused = MiniGPT(GPTConfig(**cfg_kwargs, use_fused_kernel=True)).to(device)

    x_bench = torch.randint(0, 50257, (batch_size_bench, seq_len), device=device)

    ms_vanilla, mem_vanilla = benchmark_forward(m_vanilla, x_bench)
    ms_fused, mem_fused = benchmark_forward(m_fused, x_bench)

    bench_rows.append({
        'seq_len': seq_len,
        'vanilla_ms': round(ms_vanilla, 3), 'fused_ms': round(ms_fused, 3),
        'speedup': round(ms_vanilla / ms_fused, 2),
        'vanilla_mem_mb': round(mem_vanilla, 1), 'fused_mem_mb': round(mem_fused, 1),
    })
    print(f'seq_len={seq_len}: vanilla={ms_vanilla:.3f}ms fused={ms_fused:.3f}ms '
          f'speedup={ms_vanilla/ms_fused:.2f}x | vanilla_mem={mem_vanilla:.1f}MB fused_mem={mem_fused:.1f}MB')

bench_df = pd.DataFrame(bench_rows)
bench_df

MiniGPT initialized: 29.97M parameters
MiniGPT initialized: 29.97M parameters
seq_len=64: vanilla=18.006ms fused=20.132ms speedup=0.89x | vanilla_mem=2490.6MB fused_mem=2490.6MB
MiniGPT initialized: 30.00M parameters
MiniGPT initialized: 30.00M parameters
seq_len=128: vanilla=37.053ms fused=41.778ms speedup=0.89x | vanilla_mem=2690.5MB fused_mem=2690.5MB
MiniGPT initialized: 30.04M parameters
MiniGPT initialized: 30.04M parameters
seq_len=256: vanilla=75.982ms fused=91.787ms speedup=0.83x | vanilla_mem=3089.1MB fused_mem=3089.1MB


,seq_len,vanilla_ms,fused_ms,speedup,vanilla_mem_mb,fused_mem_mb
0,64,18.006,20.132,0.89,2490.6,2490.6
1,128,37.053,41.778,0.89,2690.5,2690.5
2,256,75.982,91.787,0.83,3089.1,3089.1


In [30]:
import csv

log_path = 'results/attention_kernel_benchmark.csv'
bench_df.to_csv(log_path, index=False)
print(f'Saved to {log_path}')

!cp results/attention_kernel_benchmark.csv {DRIVE_DIR}/results/
print('Backed up to Drive.')

Saved to results/attention_kernel_benchmark.csv
Backed up to Drive.
